In [1]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_FLIGHT_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "flights"
    / "bts_ontime_2025_01.csv"
)

RAW_FLIGHT_PATH

PosixPath('/Users/tseringgurung/Desktop/flight-operations-intelligence/data/raw/flights/bts_ontime_2025_01.csv')

In [3]:
if not RAW_FLIGHT_PATH.exists():
    raise FileNotFoundError(
        f"Flight file not found at: {RAW_FLIGHT_PATH}"
    )

print("Flight data file found.")

Flight data file found.


In [4]:
con = duckdb.connect()

flight_preview = con.sql(
    f"""
    SELECT *
    FROM read_csv_auto(
        '{RAW_FLIGHT_PATH}',
        sample_size = 100000,
        ignore_errors = true
    )
    LIMIT 5
    """
).df()

flight_preview

,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,column109
0,2025,1,1,1,3,2025-01-01,AA,19805,AA,N104NN,...,None,None,None,None,None,None,None,None,None,None
1,2025,1,1,2,4,2025-01-02,AA,19805,AA,N110AN,...,None,None,None,None,None,None,None,None,None,None
2,2025,1,1,3,5,2025-01-03,AA,19805,AA,N106NN,...,None,None,None,None,None,None,None,None,None,None
3,2025,1,1,4,6,2025-01-04,AA,19805,AA,N117AN,...,None,None,None,None,None,None,None,None,None,None
4,2025,1,1,5,7,2025-01-05,AA,19805,AA,N104NN,...,None,None,None,None,None,None,None,None,None,None


In [5]:
row_count = con.sql(
    f"""
    SELECT COUNT(*) AS total_rows
    FROM read_csv_auto(
        '{RAW_FLIGHT_PATH}',
        sample_size = 100000,
        ignore_errors = true
    )
    """
).df()

row_count

,total_rows
0,539747


In [6]:
flight_columns = con.sql(
    f"""
    DESCRIBE
    SELECT *
    FROM read_csv_auto(
        '{RAW_FLIGHT_PATH}',
        sample_size = 100000,
        ignore_errors = true
    )
    """
).df()

flight_columns

,column_name,column_type,null,key,default,extra
0,Year,BIGINT,YES,None,None,None
1,Quarter,BIGINT,YES,None,None,None
2,Month,BIGINT,YES,None,None,None
3,DayofMonth,BIGINT,YES,None,None,None
4,DayOfWeek,BIGINT,YES,None,None,None
...,...,...,...,...,...,...
105,Div5TotalGTime,VARCHAR,YES,None,None,None
106,Div5LongestGTime,VARCHAR,YES,None,None,None
107,Div5WheelsOff,VARCHAR,YES,None,None,None
108,Div5TailNum,VARCHAR,YES,None,None,None


In [7]:
flights_sample = con.sql(
    f"""
    SELECT *
    FROM read_csv_auto(
        '{RAW_FLIGHT_PATH}',
        sample_size = 100000,
        ignore_errors = true
    )
    USING SAMPLE 100000 ROWS
    """
).df()

flights_sample.shape

(100000, 110)

In [8]:
flights_sample.head()

,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,column109
0,2025,1,1,2,4,2025-01-02,AA,19805,AA,N805NN,...,None,None,None,None,None,None,None,None,None,None
1,2025,1,1,19,7,2025-01-19,G4,20368,G4,274NV,...,None,None,None,None,None,None,None,None,None,None
2,2025,1,1,2,4,2025-01-02,AA,19805,AA,N860NN,...,None,None,None,None,None,None,None,None,None,None
3,2025,1,1,5,7,2025-01-05,NK,20416,NK,N631NK,...,None,None,None,None,None,None,None,None,None,None
4,2025,1,1,20,1,2025-01-20,DL,19790,DL,N131DU,...,None,None,None,None,None,None,None,None,None,None


In [9]:
flights_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Columns: 110 entries, Year to column109
dtypes: Int64(7), datetime64[us](1), float64(30), int64(17), object(55)
memory usage: 84.6+ MB


In [10]:
profile = pd.DataFrame(
    {
        "column": flights_sample.columns,
        "dtype": flights_sample.dtypes.astype(str).values,
        "missing_count": flights_sample.isna().sum().values,
        "missing_pct": (
            flights_sample.isna().mean().mul(100).round(2).values
        ),
        "unique_values": flights_sample.nunique(dropna=True).values,
    }
)

profile.sort_values(
    by="missing_pct",
    ascending=False
).head(25)

,column,dtype,missing_count,missing_pct,unique_values
109,column109,object,100000,100.0,0
85,Div3Airport,object,100000,100.0,0
91,Div3WheelsOff,object,100000,100.0,0
90,Div3LongestGTime,object,100000,100.0,0
89,Div3TotalGTime,object,100000,100.0,0
88,Div3WheelsOn,object,100000,100.0,0
87,Div3AirportSeqID,object,100000,100.0,0
86,Div3AirportID,object,100000,100.0,0
84,Div2TailNum,object,100000,100.0,0
93,Div4Airport,object,100000,100.0,0


In [11]:
important_keywords = [
    "FlightDate",
    "Carrier",
    "Origin",
    "Dest",
    "CRSDepTime",
    "DepTime",
    "DepDelay",
    "CRSArrTime",
    "ArrTime",
    "ArrDelay",
    "Cancelled",
    "Diverted",
    "Distance",
    "WeatherDelay",
    "CarrierDelay",
    "NASDelay",
    "LateAircraftDelay",
]

available_important_columns = [
    column
    for column in flights_sample.columns
    if any(
        keyword.lower() in column.lower()
        for keyword in important_keywords
    )
]

available_important_columns

['FlightDate',
 'OriginAirportID',
 'OriginAirportSeqID',
 'OriginCityMarketID',
 'Origin',
 'OriginCityName',
 'OriginState',
 'OriginStateFips',
 'OriginStateName',
 'OriginWac',
 'DestAirportID',
 'DestAirportSeqID',
 'DestCityMarketID',
 'Dest',
 'DestCityName',
 'DestState',
 'DestStateFips',
 'DestStateName',
 'DestWac',
 'CRSDepTime',
 'DepTime',
 'DepDelay',
 'DepDelayMinutes',
 'DepTimeBlk',
 'CRSArrTime',
 'ArrTime',
 'ArrDelay',
 'ArrDelayMinutes',
 'ArrTimeBlk',
 'Cancelled',
 'Diverted',
 'Distance',
 'DistanceGroup',
 'CarrierDelay',
 'WeatherDelay',
 'NASDelay',
 'LateAircraftDelay',
 'FirstDepTime',
 'DivReachedDest',
 'DivArrDelay',
 'DivDistance']

In [13]:
flights_sample[available_important_columns].head()

,FlightDate,OriginAirportID,OriginAirportSeqID,OriginCityMarketID,Origin,OriginCityName,OriginState,OriginStateFips,OriginStateName,OriginWac,...,Distance,DistanceGroup,CarrierDelay,WeatherDelay,NASDelay,LateAircraftDelay,FirstDepTime,DivReachedDest,DivArrDelay,DivDistance
0,2025-01-02,11298,1129806,30194,DFW,"Dallas/Fort Worth, TX",TX,48,Texas,74,...,852.0,4,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
1,2025-01-19,14696,1469608,34696,SBN,"South Bend, IN",IN,18,Indiana,42,...,973.0,4,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
2,2025-01-02,13303,1330303,32467,MIA,"Miami, FL",FL,12,Florida,33,...,650.0,3,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
3,2025-01-05,11298,1129806,30194,DFW,"Dallas/Fort Worth, TX",TX,48,Texas,74,...,731.0,3,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
4,2025-01-20,10713,1071302,30713,BOI,"Boise, ID",ID,16,Idaho,83,...,290.0,2,0.0,0.0,22.0,0.0,None,NaN,NaN,NaN


In [14]:
columns_to_check = [
    "Cancelled",
    "Diverted",
    "ArrDelay",
    "CRSDepTime",
    "Origin",
    "Dest",
    "Reporting_Airline"
]

for col in columns_to_check:
    print(f"{col}: {col in flights_sample.columns}")

Cancelled: True
Diverted: True
ArrDelay: True
CRSDepTime: True
Origin: True
Dest: True
Reporting_Airline: True


In [15]:
completed_flights = flights_sample[
    (flights_sample["Cancelled"] == 0)
    & (flights_sample["Diverted"] == 0)
    & (flights_sample["ArrDelay"].notna())
].copy()

completed_flights.shape

(96741, 110)

In [16]:
completed_flights["significant_arrival_delay"] = (
    completed_flights["ArrDelay"] >= 15
).astype(int)

In [17]:
completed_flights[
    ["ArrDelay", "significant_arrival_delay"]
].head(10)

,ArrDelay,significant_arrival_delay
0,9.0,0
1,2.0,0
2,-15.0,0
3,3.0,0
4,22.0,1
5,-25.0,0
6,24.0,1
7,-1.0,0
8,-4.0,0
10,-12.0,0


In [20]:
completed_flights["significant_arrival_delay"].value_counts()



significant_arrival_delay
0    78610
1    18131
Name: count, dtype: int64

In [21]:
completed_flights["significant_arrival_delay"].value_counts(
    normalize=True
).mul(100).round(2)

significant_arrival_delay
0    81.26
1    18.74
Name: proportion, dtype: float64

## Initial Target Distribution

After excluding cancelled, diverted, and flights with missing arrival-delay information:

- Completed flights analyzed: 96,741
- On-time / <15 min delay: 78,610 (81.26%)
- Significant arrival delay (≥15 min): 18,131 (18.74%)

The target exhibits moderate class imbalance. Therefore, model evaluation should not rely on accuracy alone. Precision, recall, F1-score, PR-AUC, and probability calibration will be considered during model evaluation.

In [22]:
flights_sample["Cancelled"].value_counts(dropna=False)

Cancelled
0.0    96967
1.0     3033
Name: count, dtype: int64

In [23]:
flights_sample["Diverted"].value_counts(dropna=False)

Diverted
0.0    99774
1.0      226
Name: count, dtype: int64

In [24]:
print(
    "Cancellation rate:",
    round(flights_sample["Cancelled"].mean() * 100, 2),
    "%"
)

print(
    "Diversion rate:",
    round(flights_sample["Diverted"].mean() * 100, 2),
    "%"
)

Cancellation rate: 3.03 %
Diversion rate: 0.23 %


In [25]:
flights_sample["Reporting_Airline"].value_counts()

Reporting_Airline
WN    19713
DL    13999
AA    13988
OO    12025
UA    11381
YX     5219
MQ     3950
OH     3907
AS     3402
B6     3312
NK     3232
F9     2866
G4     1786
HA     1220
Name: count, dtype: int64

In [26]:
flights_sample["Origin"].value_counts().head(15)

Origin
DFW    4722
DEN    4534
ATL    4397
ORD    4012
CLT    3217
PHX    2901
LAX    2841
LAS    2802
MCO    2369
SEA    2189
DCA    2157
LGA    2058
SFO    2030
BOS    1993
MIA    1854
Name: count, dtype: int64

In [27]:
flights_sample["Dest"].value_counts().head(15)

Dest
DFW    4622
DEN    4505
ATL    4408
ORD    4088
CLT    3099
PHX    3001
LAX    2833
LAS    2787
MCO    2441
DCA    2229
SEA    2191
LGA    2012
BOS    1999
SFO    1975
EWR    1898
Name: count, dtype: int64

In [28]:
carrier_delay = (
    completed_flights
    .groupby("Reporting_Airline")
    .agg(
        total_flights=("significant_arrival_delay", "size"),
        delayed_flights=("significant_arrival_delay", "sum"),
        avg_arrival_delay=("ArrDelay", "mean")
    )
    .reset_index()
)

carrier_delay["delay_rate"] = (
    carrier_delay["delayed_flights"]
    / carrier_delay["total_flights"]
    * 100
)

carrier_delay.sort_values(
    "delay_rate",
    ascending=False
).round(2)

,Reporting_Airline,total_flights,delayed_flights,avg_arrival_delay,delay_rate
4,F9,2782,729,12.71,26.20
9,OH,3544,911,13.65,25.71
5,G4,1764,430,13.18,24.38
2,B6,3243,777,6.23,23.96
10,OO,11716,2429,7.43,20.73
3,DL,13563,2597,5.45,19.15
7,MQ,3760,706,6.00,18.78
0,AA,13456,2525,4.56,18.76
11,UA,11132,1972,1.12,17.71
1,AS,3353,593,-0.50,17.69


In [31]:
row_count

,total_rows
0,539747


In [32]:
flights_jan = con.sql(
    f"""
    SELECT *
    FROM read_csv_auto(
        '{RAW_FLIGHT_PATH}',
        sample_size = 100000,
        ignore_errors = true
    )
    """
).df()

flights_jan.shape

(539747, 110)

In [33]:
memory_mb = flights_jan.memory_usage(deep=True).sum() / 1024**2

print(f"Memory usage: {memory_mb:.2f} MB")

Memory usage: 1320.51 MB


In [34]:
del flights_jan

In [35]:
import gc
gc.collect()

0

In [36]:
con.execute(
    f"""
    CREATE OR REPLACE VIEW flights_jan AS
    SELECT *
    FROM read_csv_auto(
        '{RAW_FLIGHT_PATH}',
        sample_size = 100000,
        ignore_errors = true
    )
    """
)

In [37]:
con.sql("""
SELECT COUNT(*) AS total_flights
FROM flights_jan
""").df()

,total_flights
0,539747
